# ViT pruning benchmark — parametrized via Papermill, Wanda-calibrated

Same algorithms and hyperparameters as the SmolLM notebook, with Wanda-style
activation calibration (input feature norms collected from a small calibration
set). Reports:
- baseline ImageNet accuracy
- accuracy after pruning **one** target layer (the largest Linear in the model)
- accuracy after pruning **all** non-head Linear layers
- per-layer relative error CSV

In [ ]:
# Default parameters. Papermill injects values from run_vit_experiments.py.
ALGORITHM = 'block_wanda'
BLOCK_ROWS = 1
BLOCK_COLS = 8
SPARSITY = 0.5
MAX_ITER = 10
RANDOM_SWAPS = 10
SWAP_FRACTION = 1/30
SORT_START = False
N_CALIB_SAMPLES = 128       # number of images for activation calibration
SKIP_ACCURACY = False
SKIP_SINGLE_LAYER_ACC = True   # only runs all-layers eval
INNER_REFINE = 2
# Very small models
# MODEL_NAME = "test_vit3.r160_in1k"

# Bigger models
MODEL_NAME = "vit_wee_patch16_reg1_gap_256.sbb_in1k"
# MODEL_NAME = "vit_medium_patch16_reg4_gap_256.sbb_in1k"

# Or others at https://huggingface.co/timm/vit_wee_patch16_reg1_gap_256.sbb_in1k

def _safe_name(s):
    return s.replace("/", "_").replace(":", "_")

CALIB_CACHE_PATH = f"calibration_cache_{_safe_name(MODEL_NAME)}_n{N_CALIB_SAMPLES}.pt"
BASELINE_ACC_CACHE_PATH = f"baseline_acc_cache_{_safe_name(MODEL_NAME)}.pt"

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"

IMAGENET_PATH = "/mnt/nvme/imagenet"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import time
import random
from functools import partial

import numpy as np
import torch
import torch.nn as nn

import timm
from tqdm.notebook import tqdm


def random_seed(seed=42, rank=0):
    torch.manual_seed(seed + rank)
    np.random.seed(seed + rank)
    random.seed(seed + rank)

random_seed(47)

device = torch.device("cuda")
amp_autocast = partial(torch.autocast, device_type=device.type, dtype=torch.float16)

In [17]:
def build_model():
    """Create a fresh pretrained model (used to reset between single-layer
    and all-layer pruning runs)."""
    m = timm.create_model(MODEL_NAME, pretrained=True)
    m.cuda()
    return m

model = build_model()
print(f"Model: {MODEL_NAME}, params: {sum(p.numel() for p in model.parameters()):,}")

Model: test_vit3.r160_in1k, params: 930,280


In [18]:
def validate(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for input, target in tqdm(val_loader):
            with amp_autocast():
                output = model(input)
                correct += (target == output.argmax(dim=1)).sum().item()
                total += target.numel()
    acc = correct / total
    print(f"val acc: {acc:.4f}")
    return acc

In [19]:
data_config = timm.data.resolve_data_config(model=model)
print(data_config)

# val_loader: only needed for accuracy validation
if not SKIP_ACCURACY:
    val_dataset = timm.data.create_dataset(
        name="imagenet",
        split="validation",
        root=IMAGENET_PATH,
    )
    val_loader = timm.data.create_loader(
        val_dataset,
        input_size=data_config['input_size'],
        batch_size=128,
        use_prefetcher=True,
        interpolation=data_config['interpolation'],
        mean=data_config['mean'],
        std=data_config['std'],
        num_workers=8,
        crop_pct=data_config["crop_pct"],
        crop_mode=data_config['crop_mode'],
        crop_border_pixels=False,
        pin_memory=True,
        device=device,
    )
else:
    val_loader = None
    print("Skipping val_loader creation (SKIP_ACCURACY=True)")

# calib_loader: only needed if calibration cache is cold
if not os.path.exists(CALIB_CACHE_PATH):
    train_dataset = timm.data.create_dataset(
        name="imagenet",
        split="train",
        root=IMAGENET_PATH,
    )
    calib_loader = timm.data.create_loader(
        train_dataset,
        input_size=data_config['input_size'],
        batch_size=64,
        use_prefetcher=True,
        interpolation="random",
        mean=data_config['mean'],
        std=data_config['std'],
        num_workers=8,
        crop_pct=data_config["crop_pct"],
        crop_mode=data_config['crop_mode'],
        crop_border_pixels=False,
        pin_memory=True,
        device=device,
        is_training=True,
    )
else:
    calib_loader = None
    print(f"Skipping calib_loader creation (cache hit at {CALIB_CACHE_PATH})")

{'input_size': (3, 160, 160), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.95, 'crop_mode': 'center'}


## Baseline accuracy (no pruning)

In [20]:
if SKIP_ACCURACY:
    baseline_acc = float("nan")
    print("Skipping baseline accuracy (SKIP_ACCURACY=True)")
elif os.path.exists(BASELINE_ACC_CACHE_PATH):
    cache = torch.load(BASELINE_ACC_CACHE_PATH)
    assert cache["model_name"] == MODEL_NAME, \
        f"Baseline cache is for {cache['model_name']}, not {MODEL_NAME}. Delete {BASELINE_ACC_CACHE_PATH}."
    baseline_acc = cache["baseline_acc"]
    print(f"Loaded cached baseline accuracy: {baseline_acc:.4f}")
else:
    baseline_acc = validate(model, val_loader)
    torch.save({"model_name": MODEL_NAME, "baseline_acc": baseline_acc}, BASELINE_ACC_CACHE_PATH)
    print(f"Cached baseline accuracy {baseline_acc:.4f} to {BASELINE_ACC_CACHE_PATH}")

  0%|          | 0/391 [00:00<?, ?it/s]

val acc: 0.5692
Cached baseline accuracy 0.5692 to baseline_acc_cache_test_vit3.r160_in1k.pt


## Identify all prunable layers + the single largest one

We collect every `nn.Linear` layer outside the classification head, then pick
the one with the largest weight matrix as the *single-layer ablation target*.
This is a deterministic, model-agnostic choice and matches intuition that
pruning matters most for the biggest layer.

In [21]:
def list_prunable_layers(model):
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, nn.Linear) and "head" not in n]


prunable = list_prunable_layers(model)
print(f"Prunable layers: {len(prunable)}")
for n, m in prunable[:5]:
    print(f"  {n}  {tuple(m.weight.shape)}")
print("  ...")

largest_name, largest_mod = max(prunable, key=lambda nm: nm[1].weight.numel())
print(f"\nSingle-layer target: {largest_name}, shape {tuple(largest_mod.weight.shape)}")

Prunable layers: 41
  blocks.0.attn.qkv  (288, 96)
  blocks.0.attn.proj  (96, 96)
  blocks.0.mlp.fc1  (192, 96)
  blocks.0.mlp.fc2  (96, 192)
  blocks.1.attn.qkv  (288, 96)
  ...

Single-layer target: blocks.0.attn.qkv, shape (288, 96)


## Wanda calibration: collect per-feature input norms

We run `N_CALIB_SAMPLES` images through the model with forward hooks attached
to every prunable Linear layer. Each hook accumulates the per-feature mean of
squared input activations. The result, `input_sq_norms[name]`, is a vector of
shape `(n_features,)` — one number per input column of the layer's weight.

Wanda score is then `(|W| * sqrt(input_sq_norms)).square()`, matching the
SmolLM helpers exactly.

In [22]:
def collect_input_sq_norms(model, calib_loader, n_samples):
    """Attach hooks to every prunable Linear, run a forward pass on
    n_samples calibration images, return {layer_name: input_sq_norms tensor}."""
    accumulators = {}
    counts = {}
    handles = []

    def make_hook(name):
        def hook(module, inputs, _output):
            X = inputs[0].detach().float()         # (..., n_features)
            X = X.reshape(-1, X.shape[-1])         # (N, n_features)
            sq_mean = X.square().mean(dim=0)       # (n_features,)
            if name not in accumulators:
                accumulators[name] = sq_mean.clone()
                counts[name] = 1
            else:
                accumulators[name] += sq_mean
                counts[name] += 1
        return hook

    for name, module in list_prunable_layers(model):
        handles.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    seen = 0
    try:
        with torch.no_grad():
            for x, _ in tqdm(calib_loader, desc="Calibrating"):
                with amp_autocast():
                    model(x)
                seen += x.size(0)
                if seen >= n_samples:
                    break
    finally:
        for h in handles:
            h.remove()

    norms = {name: accumulators[name] / counts[name] for name in accumulators}
    return norms, seen


import os
if os.path.exists(CALIB_CACHE_PATH):
    print(f"Loading cached calibration from {CALIB_CACHE_PATH}")
    cache = torch.load(CALIB_CACHE_PATH)
    assert cache["model_name"] == MODEL_NAME, \
        f"Cache is for {cache['model_name']}, but you're running {MODEL_NAME}. Delete {CALIB_CACHE_PATH} to recompute."
    input_sq_norms = {k: v.cuda() for k, v in cache["input_sq_norms"].items()}
    n_calib_seen = cache["n_calib_seen"]
    print(f"Loaded norms for {len(input_sq_norms)} layers from {n_calib_seen} cached images")
else:
    if calib_loader is None:
        raise RuntimeError(
            f"No calibration cache at {CALIB_CACHE_PATH} and calib_loader is None. "
            "If you set SKIP_ACCURACY=True, you also need to ensure calib_loader gets created. "
            "Check cell 6's conditional logic."
        )
    input_sq_norms, n_calib_seen = collect_input_sq_norms(model, calib_loader, N_CALIB_SAMPLES)
    print(f"\nCollected input norms for {len(input_sq_norms)} layers from {n_calib_seen} images")
    torch.save({
        "model_name": MODEL_NAME,
        "input_sq_norms": {k: v.cpu() for k, v in input_sq_norms.items()},
        "n_calib_seen": n_calib_seen,
        "n_calib_samples_requested": N_CALIB_SAMPLES,
    }, CALIB_CACHE_PATH)
    print(f"Cached to {CALIB_CACHE_PATH}")

# Sanity check
for name, module in list_prunable_layers(model):
    assert name in input_sq_norms, f"Missing norms for {name}"
    n_in = module.weight.shape[1]
    assert input_sq_norms[name].shape == (n_in,), \
        f"{name}: expected ({n_in},), got {tuple(input_sq_norms[name].shape)}"
print("All shapes match.")

Calibrating:   0%|          | 0/20018 [00:00<?, ?it/s]


Collected input norms for 41 layers from 128 images
Cached to calibration_cache_test_vit3.r160_in1k_n128.pt
All shapes match.


## Pruning helpers — Wanda-scored, dispatched by ALGORITHM

The score matrix is the Wanda metric, $(|W| \cdot \sqrt{||X||^2})^2$, matching
SmolLM. Pruning is performed on **the score matrix**, but the resulting
permutation and mask are then applied to the **actual weights** (which keeps
the model's forward pass numerically correct).

In [ ]:
from tetris import (
    tetris_pruning,
    original_tetris_pruning,
    random_swaps,
    sort_columns_by_norm,
    random_permutation_pruning,
    block_sparsity_pruning,
)


def wanda_score(weight, in_sq_norms):
    """Compute Wanda score matrix matching SmolLM's do_*_block_wanda helpers.

    weight: (out, in) PyTorch tensor (any dtype, any device)
    in_sq_norms: (in,) tensor of per-feature mean squared activations
    Returns a float32 score matrix of shape (out, in).
    """
    norm = in_sq_norms.to(weight.device).float().sqrt() + 1e-8
    return (weight.detach().float() * norm).square()


def _apply_perm_and_mask(weight, score, perm, mask):
    """Apply perm+mask to weights and return (pruned_weight, score_pruned_sum, score_baseline_sum).

    score_pruned_sum    = sum of score values that ended up pruned under the algorithm's perm+mask
    score_baseline_sum  = sum of score values that would be pruned by plain block pruning (no perm)
    """
    if mask is None:
        permuted_score = score[:, perm]
        _, mask = block_sparsity_pruning(
            permuted_score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
        )

    # Score-space metric: how much score did we throw away?
    permuted_score = score[:, perm]
    score_pruned_sum = permuted_score[mask == 0].sum().item()

    # Baseline (no permutation, plain block pruning of the score)
    _, baseline_mask = block_sparsity_pruning(
        score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
    )
    score_baseline_sum = score[baseline_mask == 0].sum().item()

    # Apply the perm+mask to the actual weights and unpermute
    W_permuted = weight[:, perm]
    W_pruned = W_permuted * mask.to(W_permuted.device)
    inv_perm = torch.argsort(perm)
    W_unpermuted = W_pruned[:, inv_perm].to(weight.dtype)

    return W_unpermuted, score_pruned_sum, score_baseline_sum


def do_block_only(weight, score):
    _, mask = block_sparsity_pruning(
        score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
    )
    score_pruned_sum = score[mask == 0].sum().item()
    score_baseline_sum = score_pruned_sum  # block_only IS the baseline → 0% improvement
    W_pruned = (weight * mask.to(weight.device)).to(weight.dtype)
    return W_pruned, score_pruned_sum, score_baseline_sum


def do_our_tetris(weight, score):
    _, _, perm, *_ = tetris_pruning(
        W=score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
        max_iter=MAX_ITER, random_swaps=RANDOM_SWAPS, verbose=False, inner_refine=INNER_REFINE,
    )
    return _apply_perm_and_mask(weight, score, perm, mask=None)


def do_original_tetris(weight, score):
    _, _, perm, *_ = original_tetris_pruning(
        W=score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
        max_iter=MAX_ITER, verbose=False,
    )
    return _apply_perm_and_mask(weight, score, perm, mask=None)


def do_random_swaps(weight, score):
    extra = {}
    import inspect
    sig = inspect.signature(random_swaps)
    if 'swap_fraction' in sig.parameters:
        extra['swap_fraction'] = SWAP_FRACTION
    if 'sort_start' in sig.parameters:
        extra['sort_start'] = SORT_START

    _, mask, perm, *_ = random_swaps(
        W=score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
        max_iter=MAX_ITER, verbose=False, **extra,
    )
    return _apply_perm_and_mask(weight, score, perm, mask=mask)


def do_sort_columns_by_norm(weight, score):
    _, mask, perm = sort_columns_by_norm(
        W=score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY, verbose=False,
    )
    return _apply_perm_and_mask(weight, score, perm, mask=mask)


def do_random_permutation(weight, score):
    _, mask, perm = random_permutation_pruning(
        W=score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY, verbose=False,
    )
    return _apply_perm_and_mask(weight, score, perm, mask=mask)


def prune_layer(weight, in_sq_norms):
    """Top-level dispatch. Returns (pruned_weight, score_pruned_sum, score_baseline_sum).

    score_improvement_pct = 100 * (score_baseline_sum - score_pruned_sum) / score_baseline_sum
    """
    if ALGORITHM == 'no_prune':
        return weight.clone(), 0.0, 0.0

    score = wanda_score(weight, in_sq_norms)
    match ALGORITHM:
        case 'block_only' | 'block_wanda':
            return do_block_only(weight, score)
        case 'our_tetris':
            return do_our_tetris(weight, score)
        case 'original_tetris':
            return do_original_tetris(weight, score)
        case 'random_swaps':
            return do_random_swaps(weight, score)
        case 'sort_columns_by_norm':
            return do_sort_columns_by_norm(weight, score)
        case 'random_permutation_pruning':
            return do_random_permutation(weight, score)
        case _:
            raise ValueError(f"Unknown algorithm: {ALGORITHM}")

## Single-layer pruning

Restore a fresh model, prune only the largest layer, validate.

We **must reuse the calibration data we already collected on the unpruned
model**: re-running calibration on the fresh model would give the same norms
(it's the same pretrained checkpoint). So we can keep `input_sq_norms` and
just look up the entry for our target layer.

In [24]:
model_single = build_model()
prunable_single = list_prunable_layers(model_single)

target_name = largest_name
target_mod = dict(prunable_single)[target_name]

assert target_name in input_sq_norms, f"No calibration norms for {target_name}"
target_norms = input_sq_norms[target_name]

t0 = time.perf_counter()
W_pruned, score_pruned_sum, score_baseline_sum = prune_layer(target_mod.weight.data, target_norms)
single_layer_time = time.perf_counter() - t0

# Score-space improvement
if score_baseline_sum > 0:
    single_layer_score_improvement_pct = 100 * (score_baseline_sum - score_pruned_sum) / score_baseline_sum
else:
    single_layer_score_improvement_pct = 0.0
print(f"Single-layer score-space improvement: {single_layer_score_improvement_pct:.2f}%")


# Wanda-weighted relative error: matches SmolLM's reporting metric
W_orig = target_mod.weight.data.detach().float()
W_p = W_pruned.detach().float()
norm_dev = target_norms.to(W_orig.device).float()
single_layer_rel_error = (
    ((W_orig - W_p).square() * norm_dev).sum().item()
    / (W_orig.square() * norm_dev).sum().item()
)
single_layer_density = (W_pruned != 0).float().mean().item()
print(f"Single-layer rel error: {single_layer_rel_error:.6f}")
print(f"Single-layer density:   {single_layer_density:.4f}")
print(f"Single-layer time:      {single_layer_time:.3f}s")

# Apply the pruned weight to the model
target_mod.weight.data = W_pruned.to(target_mod.weight.device)

if SKIP_ACCURACY or SKIP_SINGLE_LAYER_ACC:
    acc_single = float("nan")
    print(f"Skipping single-layer accuracy")
else:
    acc_single = validate(model_single, val_loader)
    print(f"Accuracy after pruning ONLY {target_name}: {acc_single:.4f}")

del model_single
torch.cuda.empty_cache()

Single-layer score-space improvement: 0.00%
Single-layer rel error: 0.221742
Single-layer density:   0.5000
Single-layer time:      0.001s
Skipping single-layer accuracy


## All-layer pruning

Restore a fresh model, prune every non-head Linear layer with the chosen
algorithm, validate. Per-layer metrics are tracked along the way.

Pruning a layer doesn't change input statistics for *later* layers in a
strictly-correct way — using activations from the unpruned model is an
approximation, but it's the standard one (Wanda does the same thing). For a
fully-correct pipeline you'd recalibrate after each layer; that's much more
expensive and rarely changes the conclusion.

In [25]:
model_all = build_model()
prunable_all = list_prunable_layers(model_all)

layer_metrics = []
errors = []

t_start = time.perf_counter()

for name, module in tqdm(prunable_all, desc="Pruning layers"):
    in_sq_norms = input_sq_norms[name]

    W_orig = module.weight.data.detach().float().clone()
    norm_dev = in_sq_norms.to(W_orig.device).float()

    t0 = time.perf_counter()
    W_pruned, score_pruned_sum, score_baseline_sum = prune_layer(module.weight.data, in_sq_norms)
    layer_time = time.perf_counter() - t0   

    W_p = W_pruned.detach().float()
    rel_error = (
        ((W_orig - W_p).square() * norm_dev).sum().item()
        / (W_orig.square() * norm_dev).sum().item()
    )
    density = (W_pruned != 0).float().mean().item()
    pruned_l1 = (W_orig - W_p).abs().sum().item()
    total_l1 = W_orig.abs().sum().item()

    # Wanda-weighted masses (same as SmolLM)
    total_wanda_mass = (W_orig.square() * norm_dev).sum().item()
    retained_wanda_mass = (W_p.square() * norm_dev).sum().item()

    # Score-space improvement (matches the metric in bigger_test_simplified.ipynb)
    if score_baseline_sum > 0:
        score_improvement_pct = 100 * (score_baseline_sum - score_pruned_sum) / score_baseline_sum
    else:
        score_improvement_pct = 0.0

    errors.append(rel_error)
    layer_metrics.append({
        "layer_name": name,
        "algorithm": ALGORITHM,
        "block_rows": BLOCK_ROWS,
        "block_cols": BLOCK_COLS,
        "sparsity": SPARSITY,
        "max_iter": MAX_ITER,
        "random_swaps": RANDOM_SWAPS,
        "swap_fraction": SWAP_FRACTION,
        "sort_start": SORT_START,
        "total_time_sec": layer_time,
        "rel_error": rel_error,
        "density": density,
        "pruned_weight_l1": pruned_l1,
        "total_weight_l1": total_l1,
        "total_wanda_mass": total_wanda_mass,
        "retained_wanda_mass": retained_wanda_mass,
        "score_pruned_sum":      score_pruned_sum,
        "score_baseline_sum":    score_baseline_sum,
        "score_improvement_pct": score_improvement_pct,
        "n_rows": W_pruned.shape[0],
        "n_cols": W_pruned.shape[1],
        "baseline_accuracy":     baseline_acc,           # might be NaN if SKIP_ACCURACY
        "accuracy_single_layer": acc_single,             # NaN before cell 16 has run
        "accuracy_all_layers":   float("nan"),           # filled in below
        "single_layer_name":     target_name,
        "n_calib_samples":       n_calib_seen,
    })

    module.weight.data = W_pruned.to(module.weight.device)

execution_time = time.perf_counter() - t_start
print(f"Total pruning time: {execution_time:.2f}s")
print(f"Mean rel error:     {np.mean(errors):.6f}")

execution_time = time.perf_counter() - t_start
print(f"Total pruning time: {execution_time:.2f}s")
print(f"Mean rel error:     {np.mean(errors):.6f}")

whole_model_validation_time = float("nan")
if SKIP_ACCURACY:
    acc_all = float("nan")
    print("Skipping all-layer accuracy (SKIP_ACCURACY=True)")
else:
    t0 = time.perf_counter()
    acc_all = validate(model_all, val_loader)
    whole_model_validation_time = time.perf_counter() - t0
    print(f"All-layer validation time: {whole_model_validation_time:.2f}s")
    print(f"Accuracy after pruning ALL layers: {acc_all:.4f}")

Pruning layers:   0%|          | 0/41 [00:00<?, ?it/s]

Total pruning time: 0.04s
Mean rel error:     0.234686
Total pruning time: 0.04s
Mean rel error:     0.234686


  0%|          | 0/391 [00:00<?, ?it/s]

val acc: 0.0024
All-layer validation time: 43.81s
Accuracy after pruning ALL layers: 0.0024


## Save metrics & glue results for Papermill

In [ ]:
import scrapbook as sb
import pandas as pd

# Patch model-level accuracy that was only knowable after the all-layer loop
for row in layer_metrics:
    row["accuracy_all_layers"] = acc_all


csv_name = (
    f"layer_metrics_{ALGORITHM}"
    f"_{BLOCK_ROWS}x{BLOCK_COLS}"
    f"_iters_{MAX_ITER}"
    f"_swaps_{RANDOM_SWAPS}"
    f"_innerref_{INNER_REFINE}"
    f"_swapfrac_{SWAP_FRACTION:.4f}"
    f"_sortstart_{SORT_START}"
    f"_sparsity_{SPARSITY}.csv"
)
csv_dir = f"benchmark_csvs_vit_{_safe_name(MODEL_NAME)}"
os.makedirs(csv_dir, exist_ok=True)
csv_path = os.path.join(csv_dir, csv_name)
os.makedirs(csv_dir, exist_ok=True)
csv_path = os.path.join(csv_dir, csv_name)
df_metrics = pd.DataFrame(layer_metrics)
df_metrics.to_csv(csv_path, index=False)

# Keep scraps for backward compat with run_vit_experiments.py's summary table
sb.glue("baseline_accuracy", baseline_acc)
sb.glue("accuracy_single_layer", acc_single)
sb.glue("accuracy_all_layers", acc_all)
sb.glue("single_layer_name", target_name)
sb.glue("single_layer_rel_error", single_layer_rel_error)
sb.glue("relative_errors", errors)
sb.glue("execution_time", execution_time)
sb.glue("n_calib_samples", n_calib_seen)
sb.glue("layer_metrics_csv", csv_path)
sb.glue("whole_model_validation_time", whole_model_validation_time)

/home/jmuravska/miniconda3/envs/diplomovka/lib/python3.11/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)
